# Learning Embeddings — Smokers/Friends/Cancer FOL API

This notebook rewrites the classic smokers-friends-cancer example with the
**Signature / Formula / Interpretation** API.

## Problem

We model 14 people, named `A` through `N`, split into two groups.  Within each
group, the notebook provides observed facts about friendship, smoking, and
cancer. Each person is represented by a learnable embedding vector.  Neural predicates
then map these embeddings to fuzzy truth values:

- `Smokes(x)` — whether person `x` smokes;
- `Cancer(x)` — whether person `x` has cancer;
- `Friends(x, y)` — whether `x` and `y` are friends.

The goal is to learn the person embeddings and predicate networks so that the
observed facts and logical rules are satisfied as much as possible.

> Note: the code sets `embedding_size = 15`, so this version learns 15-D embeddings.  

## Knowledge base

The KB contains fact groups and rule formulas.

| Type | Content |
|------|---------|
| Facts | Positive and negative facts for `Friends`, `Smokes`, and `Cancer` |
| Rules | Anti-reflexive friendship, symmetric friendship, existence of a friend, smoking propagation through friendship, and smoking/cancer implications |

The following query formulas are evaluated during training but are **not** included
in the KB:

$$
\phi_1 \mid \forall x : \; (Cancer(x) \to Smokes(x))
$$

$$
\phi_2 \mid \forall(x, y) : \; (Cancer(x) \;\&\; Cancer(y) \to Friends(x, y))
$$

Parentheses in quantified expressions such as `∀(x, y)` are intentionally kept,
because they denote diagonal quantification in this notebook.

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Signature

The signature defines the symbolic vocabulary of the domain.

It contains:

- one sort: `Person`;
- constants `A` through `N`;
- variables `x` and `y`;
- unary predicates `Smokes` and `Cancer`;
- the binary predicate `Friends`.

The person names are uppercase single letters so that the parser treats them as
constants rather than variables.

In [ ]:
import sltn
from sltn import Signature, Interpretation, KB

embedding_size = 15

sig    = Signature("smokers")
sig.sort("Person")
sig.predicate("Smokes",  ["Person"])          # smokes(x)
sig.predicate("Cancer",  ["Person"])          # cancer(x)
sig.predicate("Friends", ["Person", "Person"])  # friends(x, y)

# Constants: uppercase A-N so the parser treats them as Cst (not Var)
for person in 'ABCDEFGHIJKLMN':
    sig.constant(person, "Person")

sig.variable("x", "Person")
sig.variable("y", "Person")

print(sig)

## 2. Formulas

This section defines the facts, rules, and query formulas.

### Facts

Fact formulas are closed sentences that refer directly to named constants, such as:

$$
Friends(A, B), \quad Smokes(A), \quad Cancer(E).
$$

Negative facts use logical negation:

$$
\lnot Friends(A, C), \quad \lnot Smokes(B), \quad \lnot Cancer(D).
$$

### Rules

The rule formulas are universally quantified sentences over `x` and `y`.  They
encode:

- anti-reflexive friendship;
- symmetric friendship;
- existence of at least one friend;
- propagation of smoking through friendship;
- smoking implying cancer;
- non-smoking implying non-cancer.

In [ ]:
# Person names use uppercase single letters (A–N) so that the FOL formula
# parser correctly treats them as constants (Cst) rather than variables (Var).
friends = [('A','B'),('A','E'),('A','F'),('A','G'),('B','C'),('C','D'),
           ('E','F'),('G','H'),('I','J'),('J','M'),('K','L'),('M','N')]
smokes = ['A', 'E', 'F', 'G', 'J', 'N']
cancer = ['A', 'E']
g1 = set('ABCDEFGH')
g2 = set('IJKLMN')
g  = g1 | g2


The next cell lists the observed facts. The friendship facts are specified within two groups.  Smoking and cancer facts are specified per person.

In [ ]:
# --- Positive fact groups ---
kb_friends_pos = KB([sig.parse(f"Friends({u}, {v})") for u, v in friends])
kb_smokes_pos  = KB([sig.parse(f"Smokes({p})") for p in smokes])
kb_cancer_pos  = KB([sig.parse(f"Cancer({p})") for p in cancer])

# --- Negative fact groups (separate from positives to keep equal weight in the outer SatAgg) ---
kb_friends_neg = KB([*[sig.parse(f"¬Friends({u}, {v})") for u in g1 for v in g1 if (u, v) not in friends and u < v],
                     *[sig.parse(f"¬Friends({u}, {v})") for u in g2 for v in g2 if (u, v) not in friends and u < v]])
kb_smokes_neg  = KB([sig.parse(f"¬Smokes({p})") for p in g if p not in smokes])
kb_cancer_neg  = KB([sig.parse(f"¬Cancer({p})") for p in g1 if p not in cancer])

# --- Rule axioms (each rule is its own top-level entry for equal weighting) ---
phi_anti_refl        = sig.parse("∀ x: ¬Friends(x, x)")
phi_sym              = sig.parse("∀ x, y: (Friends(x, y) -> Friends(y, x))")
phi_has_friend       = sig.parse("∀ x: ∃ y: Friends(x, y)")
phi_spread           = sig.parse("∀ x, y: ((Friends(x, y) & Smokes(x)) -> Smokes(y))")
phi_smoke_cancer     = sig.parse("∀ x: (Smokes(x) -> Cancer(x))")
phi_nosmoke_nocancer = sig.parse("∀ x: (¬Smokes(x) -> ¬Cancer(x))")

In [ ]:
phi_kb = KB([
    kb_friends_pos, kb_friends_neg,
    kb_smokes_pos,  kb_smokes_neg,
    kb_cancer_pos,  kb_cancer_neg,
    phi_anti_refl, phi_sym, phi_has_friend,
    phi_spread, phi_smoke_cancer, phi_nosmoke_nocancer,
])

# Query formulas — closed sentences evaluated on demand (not in KB)
phi_q1 = sig.parse("∀ x: (Cancer(x) -> Smokes(x))")
phi_q2 = sig.parse("∀ x, y: ((Cancer(x) & Cancer(y)) -> Friends(x, y))")

print(f"phi_kb clauses: {len(phi_kb.clauses)}")
print(f"  friends_pos: {len(kb_friends_pos.clauses)}, friends_neg: {len(kb_friends_neg.clauses)}")
print(f"  smokes_pos: {len(kb_smokes_pos.clauses)}, smokes_neg: {len(kb_smokes_neg.clauses)}")
print(f"  cancer_pos: {len(kb_cancer_pos.clauses)}, cancer_neg: {len(kb_cancer_neg.clauses)}")

## 3. Neural models and embeddings

Each person receives a learnable embedding vector.

The predicates are grounded by multilayer perceptrons:

- `Smokes` takes one person embedding and returns a truth value;
- `Cancer` takes one person embedding and returns a truth value;
- `Friends` takes two person embeddings and returns a truth value.

Each MLP ends with a sigmoid activation, so all predicate outputs lie in `[0, 1]`.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, layer_sizes):
        super().__init__()
        self.elu     = torch.nn.ELU()
        self.sigmoid = torch.nn.Sigmoid()
        self.layers  = torch.nn.ModuleList([
            torch.nn.Linear(layer_sizes[i-1], layer_sizes[i])
            for i in range(1, len(layer_sizes))
        ])

    def forward(self, *xs):
        x = torch.cat(list(xs), dim=-1) if len(xs) > 1 else xs[0]
        for layer in self.layers[:-1]: x = self.elu(layer(x))
        return self.sigmoid(self.layers[-1](x))


# Instantiate neural models — will be bound to the interpretation in section 5
smokes_net  = MLP([embedding_size, 16, 16, 1])
cancer_net  = MLP([embedding_size, 16, 16, 1])
friends_net = MLP([embedding_size * 2, 16, 16, 1])

embeddings = {
    person: torch.rand(embedding_size, device=device, requires_grad=True)
    for person in g
}

## 4. Interpretation

The interpretation connects the symbolic signature to concrete tensors, neural modules, and fuzzy logic operators.

In this notebook:

- `Person` is interpreted as a vector type of size `embedding_size`;
- constants `A` through `N` are grounded by learnable person embeddings;
- `Smokes`, `Cancer`, and `Friends` are grounded by the MLPs;
- the logical connectives and quantifiers are assigned differentiable fuzzy
  semantics.

When a fact formula mentions a constant such as `A`, the interpretation resolves
that constant to its current embedding tensor.

In [ ]:
from sltn import Logic
from sltn.interpretation.operators import aggregator, conjunction, disjunction, implication, negation

stable = True
EPS = 1e-12

# Custom logic operators
logic = Logic()
logic["∧"] = conjunction.AndProd(stable=stable, eps=EPS)
logic["∨"] = disjunction.OrProbSum(stable=stable, eps=EPS)
logic["¬"] = negation.NotStandard()
logic["→"] = implication.ImpliesReichenbach(stable=stable, eps=EPS)
logic["∀"] = aggregator.AggregPMeanError(p=2, stable=stable, eps=EPS)
logic["∃"] = aggregator.AggregPMean(p=2, stable=stable, eps=EPS)
logic["SatAgg"] = aggregator.AggregPMeanError(p=2, stable=stable, eps=EPS)

interp = Interpretation(sig, logic=logic).to(device)

interp["Person"] = sltn.Type("Person", shape=(embedding_size,))

# Bind neural models
interp["Smokes"]  = smokes_net
interp["Cancer"]  = cancer_net
interp["Friends"] = friends_net

# Bind learnable person embeddings as constants
for person, emb in embeddings.items():
    interp[person] = emb

print(interp)

## 5. Training loop

Training optimizes the satisfaction of the KB for 2000 epochs using Adam.

At each epoch:

1. the current person embeddings are stacked and assigned to `x` and `y`;
2. the KB clauses are evaluated under the current interpretation;
3. `kb_backward` computes gradients through the logical satisfaction objective;
4. gradients are clipped for stability;
5. the optimizer updates the predicate networks and embeddings.

The existential quantifier uses a curriculum:

- epochs 0–200 use a lenient aggregation setting;
- later epochs use a stricter setting to encourage the “everyone has a friend”
  rule.

The query formulas are evaluated periodically under the same interpretation, but
they are not part of the training KB.

In [ ]:
params = (
    list(interp["Smokes"].parameters()) +
    list(interp["Cancer"].parameters()) +
    list(interp["Friends"].parameters()) +
    list(embeddings.values())
)
optimizer = torch.optim.Adam(params, lr=0.001)


In [ ]:
from sltn import kb_describe, kb_backward

kb_dict = kb_describe(phi_kb)


for epoch in range(2000):
    # Curriculum: start with p=1 (lenient ∃) then switch to p=6 (strict) after epoch 200
    p = 1 if epoch <= 200 else 6
    interp.logic.Exists.p = p
    interp.logic.SatAgg.p = p
    interp.logic.Forall.p = p

    # Rebuild embedding stacks so gradients flow through the latest values.
    # interp(phi_kb) resolves x and y from interp["x"] / interp["y"] automatically.
    interp["x"] = torch.stack([embeddings[person] for person in sorted(g)])
    interp["y"] = interp["x"]

    optimizer.zero_grad()
    stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
    # Gradient clipping: guards against rare large-gradient steps near the p-curriculum switch.
    torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
    optimizer.step()

    if epoch % 100 == 0:
        losses_str = " | ".join(
            f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
        )
        w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
        print(f"epoch {epoch:4d} | [{losses_str}] | α=[{w_str}]")

        if epoch % 100 == 0:
            interp["x"] = torch.stack([embeddings[person] for person in sorted(g)])
            interp["y"] = interp["x"]
            with torch.no_grad():
                phi1_val = interp(phi_q1).value
                phi2_val = interp(phi_q2).value
            print(
                f"| Phi1 {phi1_val.item():.3f} | Phi2 {phi2_val.item():.3f}"
            )


## 6. Visualizing results

The first figure shows the observed facts.

- Values near 1 indicate known true facts.
- Values near 0 indicate known false facts.
- White cells indicate unknown or unused entries.

The second figure shows the learned predicate values after training for:

- `Smokes(x)`;
- `Cancer(x)`;
- `Friends(x, y)` within each group.

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

persons_sorted = sorted(g)
g1s = sorted(g1)
g2s = sorted(g2)

def plt_heatmap(df, vmin=None, vmax=None):
    plt.pcolor(df, vmin=vmin, vmax=vmax)
    plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
    plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns)
    plt.colorbar()

# --- Ground-truth facts ---
df_facts_sc = pd.DataFrame(
    np.array([[(p in smokes), (p in cancer) if p in g1 else math.nan] for p in persons_sorted]),
    columns=["Smokes(x)", "Cancer(x)"], index=persons_sorted)
df_facts_f1 = pd.DataFrame(
    np.array([[float((x, y) in friends or (y, x) in friends) if x != y else math.nan for x in g1s] for y in g1s]),
    index=g1s, columns=g1s)
df_facts_f2 = pd.DataFrame(
    np.array([[float((x, y) in friends or (y, x) in friends) if x != y else math.nan for x in g2s] for y in g2s]),
    index=g2s, columns=g2s)

plt.figure(figsize=(14, 3))
plt.subplot(131); plt_heatmap(df_facts_sc, 0, 1); plt.title("Facts: Smokes(x), Cancer(x)")
plt.subplot(132); plt_heatmap(df_facts_f1, 0, 1); plt.title("Facts: Friends(x,y) Group 1")
plt.subplot(133); plt_heatmap(df_facts_f2, 0, 1); plt.title("Facts: Friends(x,y) Group 2")
plt.suptitle("Ground-truth facts (white = unknown)")
plt.tight_layout(); plt.show()

# --- Learned predicate values ---
with torch.no_grad():
    emb_stack = torch.stack([embeddings[p] for p in persons_sorted]).to(device)
    s_vals = interp["Smokes"](emb_stack).squeeze().cpu().numpy()
    c_vals = interp["Cancer"](emb_stack).squeeze().cpu().numpy()
    # Friends is binary; compute full [14,14] matrix
    N = len(persons_sorted)
    f_vals = interp["Friends"](
        emb_stack.unsqueeze(1).expand(N, N, embedding_size).reshape(N*N, embedding_size),
        emb_stack.unsqueeze(0).expand(N, N, embedding_size).reshape(N*N, embedding_size),
    ).reshape(N, N).cpu().numpy()

df_pred_sc = pd.DataFrame(
    np.stack([s_vals, c_vals], axis=1),
    columns=["Smokes(x)", "Cancer(x)"], index=persons_sorted)

df_pred_f1 = pd.DataFrame(f_vals[:8, :8], index=g1s, columns=g1s)
df_pred_f2 = pd.DataFrame(f_vals[8:, 8:], index=g2s, columns=g2s)

df_pred_f1.iloc[range(len(g1s)), range(len(g1s))] = np.nan
df_pred_f2.iloc[range(len(g2s)), range(len(g2s))] = np.nan

plt.figure(figsize=(14, 3))
plt.subplot(131); plt_heatmap(df_pred_sc, 0, 1); plt.title("Learned: Smokes(x), Cancer(x)")
plt.subplot(132); plt_heatmap(df_pred_f1, 0, 1); plt.title("Learned: Friends(x,y) Group 1")
plt.subplot(133); plt_heatmap(df_pred_f2, 0, 1); plt.title("Learned: Friends(x,y) Group 2")
plt.suptitle("Learned predicate values after training")
plt.tight_layout(); plt.show()


## 7. Clause-level satisfaction

This plot shows the final satisfaction of each KB clause. It is useful for identifying which constraints are satisfied strongly and which remain in tension with the rest of the knowledge base.

In [ ]:
with torch.no_grad():
    interp["x"] = torch.stack([embeddings[person] for person in sorted(g)])
    interp["y"] = interp["x"]
    clause_sats = [interp(clause).value.item() for clause in phi_kb.clauses]

df_clause_sats = pd.DataFrame(
    clause_sats,
    index=[f"Clause {phi_kb.clauses[i]}"[:70] for i in range(len(phi_kb.clauses))],
    columns=["Satisfaction"]
)

plt.figure(figsize=(16, len(phi_kb.clauses) * 0.3))
plt.barh(df_clause_sats.index, df_clause_sats["Satisfaction"], color='skyblue')
plt.xlim(0, 1)
plt.xlabel("Satisfaction")
plt.title("Satisfaction of each KB clause after training")
plt.tight_layout()
plt.show()


## 8. Discussion

The learned grounding captures much of the smokers-friends-cancer structure while
handling inconsistent constraints softly.

### Friendship

The model learns high `Friends(x, y)` values for many known friend pairs and low
values for many known non-friend pairs. The symmetry rule encourages `Friends(x, y)` and `Friends(y, x)` to have similar
truth values.  The anti-reflexive rule discourages self-friendship.

### Smoking and cancer

The observed smoking and cancer facts are not perfectly consistent with the rules. For example, some people are observed as smokers without observed cancer, while the
rule
$$
\forall x : \; (Smokes(x) \to Cancer(x))
$$)
pushes smokers toward higher cancer truth values. Because the logic is differentiable and soft, the model does not need to satisfy
every clause exactly.  Instead, it finds a compromise that balances facts and rules.

### Queries
The first query,
$$
\phi_1\equiv \forall x : \; (Cancer(x) \to Smokes(x)),
$$
receives relatively high satisfaction.  Although this implication is not included
in the KB, it is supported indirectly by the learned relationship between smoking
and cancer.

The second query,

$$
\phi_2\equiv \forall(x, y) :\; (Cancer(x) \;\&\; Cancer(y) \to Friends(x, y)),
$$

receives lower satisfaction.  This is expected because cancer does not imply
friendship. Overall, this example shows how `sltn` can learn embeddings and neural predicate
groundings by optimizing soft logical satisfaction.
